# 05 - Integrated Pipeline: Clinical Triage Demo

Main integrative artifact for the Integrated AI System. This notebook ties together every component of the system and demonstrates three end-to-end runs through the agent orchestrator.

## Lineage map (prior capstone projects)

| Prior project | Concept adapted (code rebuilt) | Where in this system |
|---|---|---|
| **P2** - Data & Statistical Reasoning | Initial Data Analysis discipline, chi-square + Cramer's V, limitations/bias framing | `notebooks/01_data_exploration.ipynb` |
| **P3** - Machine Learning (K-Means RFM) | `log1p` + `StandardScaler` + scikit-learn Pipeline / ColumnTransformer | `src/preprocessing.py`, `src/ml_model.py` |
| **P4** - Deep Learning (CNN with dropout) | Fixed seed PyTorch training loop, dropout, per-slice disaggregated evaluation | `src/dl_model.py`, slice metrics in both ML and DL notebooks |
| **P5** - Generative AI (VAE) | Responsible-AI framing for generative outputs (under-claim capability, structural mitigations baked into the prompt, mandatory disclaimer) | `src/genai_explainer.py` |
| **P6** - Agentic AI (Research Brief Agent) | Plan -> Route -> Synthesise -> Evaluate -> Revise loop, Chroma + OpenAI embeddings, INSUFFICIENT EVIDENCE escape valve, refusal list, runtime caps, sha256 ingest manifest, JSONL run log | `src/rag/`, `src/agent_orchestrator.py`, `src/safeguards.py`, `outputs/run_log.jsonl` |

**Educational artifact only. Not for clinical use.**

## System architecture

![Integrated AI System architecture](../docs/architecture.png)

> Rendered from the Mermaid source in `scripts/render_architecture.py`. Re-run that script after editing the diagram to refresh `docs/architecture.png`.

<details><summary>Mermaid source (for reference)</summary>

```mermaid
flowchart TD
    ENTRY[Demo invocation<br/>run_all.py step 6 / notebook 05<br/>picks 3 demo patients + fixed clinician request<br/><i>run_all.py</i>] --> A
    A[Patient record<br/>UCI Heart Disease<br/><i>src/data_loader.py</i>] --> B[Preprocessing<br/>log1p + StandardScaler + one-hot<br/><i>src/preprocessing.py</i>]

    subgraph SCORING[Deterministic scoring]
        direction LR
        C[ML model<br/>HistGradientBoosting<br/><i>src/ml_model.py</i>]
        D[DL model<br/>PyTorch MLP<br/><i>src/dl_model.py</i>]
        E[Ensemble + tier + confidence flag<br/><i>src/decision.py</i>]
        C --> E
        D --> E
    end

    B --> C
    B --> D

    E --> F[Agent orchestrator<br/>plan / retrieve / explain / evaluate / revise<br/><i>src/agent_orchestrator.py</i>]

    subgraph INGEST[RAG ingestion - build time, idempotent]
        direction LR
        P[knowledge_base/*.md<br/>clinical guideline markdown]
        Q[Header-aware chunker<br/>split on H1/H2, max 1500 chars<br/><i>src/rag/knowledge_base.py</i>]
        R[OpenAI embeddings<br/>text-embedding-3-small<br/><i>src/rag/embeddings.py</i>]
        S[ChromaDB persistent<br/>HNSW + SQLite<br/><i>src/rag/vector_store.py</i>]
        T[sha256 manifest<br/>ingest_manifest.json<br/><i>src/rag/manifest.py</i>]
        P --> Q --> R --> S
        P -. hash check .-> T
        T -. skip unchanged .-> R
    end

    subgraph AGENTIC[Agentic loop - query time]
        direction LR
        G[RAG retriever<br/>top-k cosine over ChromaDB<br/><i>src/rag/retriever.py</i>]
        H[GenAI explainer<br/>OpenAI LLM + bracketed citations<br/><i>src/genai_explainer.py</i>]
        J[Evaluator-critic<br/>pass / score / issues<br/><i>src/agent_orchestrator.py</i>]
    end

    S -. read .-> G
    F --> G --> H
    H --> J
    J -. revise loop .-> H

    subgraph SAFETY[Safety side-channel]
        K[Refusal substring list + caps<br/><i>src/safeguards.py</i>]
    end
    F -. guardrail check .-> K

    H --> L[Clinician-facing explanation]
    K -. refusal .-> L

    subgraph AUDIT[Audit trail]
        direction LR
        M[outputs/run_log.jsonl<br/>append-only events<br/><i>src/agent_orchestrator.py</i>]
        N[docs/transcripts/<br/>per-run markdown<br/><i>src/transcripts.py</i>]
    end

    F -. log every event .-> M
    L -. persist .-> N
```

</details>


In [1]:
# Notebook bootstrap for the integrated demo: path setup plus every module used
# end-to-end (data, models, RAG retriever, orchestrator).

import sys, json
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.agent_orchestrator import run
from src.data_loader import load_heart_disease
from src.decision import load_default_scorers, score_cohort
from src.preprocessing import split_and_preprocess
from src.rag.retriever import ingest
from src.transcripts import save_transcript

TRANSCRIPTS = PROJECT_ROOT / 'docs' / 'transcripts'

## 1. Load data, models, and ingest KB

In [2]:
# Phase setup: load test data, load both scorers, and run KB ingestion.
# ingest() is idempotent (sha256 manifest) - on a re-run it returns 0 new chunks.

df = load_heart_disease()
split = split_and_preprocess(df) # uses rand_state=42 for reproducibility
ml, dl, pp = load_default_scorers(split)
ingest_summary = ingest()  # idempotent
print('ingest summary:', ingest_summary)

ingest summary: {'changed_files': [], 'chunks_added': 0, 'total_chunks': 29}


## 2. Score the entire test cohort
Lineage: P3 - 'report cohort sizes alongside the metric'. Here we report tier counts, model-agreement rate, and the resulting workload split a clinician would see.

In [3]:
# Score the entire test cohort with both models so we can pick demo patients
# and report agreement rate. agreement = |ml - dl| <= 0.20 (the high-confidence band).

cohort = score_cohort(split.X_test, ml, dl, pp)
cohort['agreement'] = (abs(cohort['ml_prob'] - cohort['dl_prob']) <= 0.20)
print('Tier counts:')
print(cohort['tier'].value_counts())
print()
print('Model-agreement rate:', round(cohort['agreement'].mean() * 100, 1), '%')
cohort.head(10)

Tier counts:
tier
high        29
low         28
moderate     4
Name: count, dtype: int64

Model-agreement rate: 73.8 %


,ml_prob,dl_prob,ensemble_prob,tier,agreement
219,0.564835,0.183387,0.374111,moderate,False
271,0.824733,0.778227,0.801480,high,True
89,0.001141,0.081391,0.041266,low,True
101,0.007796,0.088258,0.048027,low,True
67,0.060455,0.436182,0.248318,low,False
244,0.005871,0.117647,0.061759,low,True
185,0.216814,0.122087,0.169450,low,True
233,0.032227,0.508146,0.270186,low,False
168,0.990341,0.624691,0.807516,high,False
197,0.068564,0.394545,0.231554,low,False


<!-- chart-narration -->
**Reading the cohort scoring summary.** We score the entire held-out test set with both models. Three outputs to read:
- **Tier counts** (`high` / `moderate` / `low`): how many patients fall into each risk band using thresholds 0.3 and 0.7. The mix should look broadly clinical, not all-one-tier.
- **Ensemble probability (`ensemble_prob`)**: the final combined risk score used for triage, computed as a weighted average of the two model probabilities (default 50/50). We show it because this is the score that drives tier assignment and workload routing; values near 0.30 or 0.70 are borderline and should receive extra clinical caution.
- **Agreement rate**: fraction of patients where |ml_prob - dl_prob| <= 0.20. Higher is better; this is the empirical basis for using `confidence='high'` as a routing signal in the agentic step.

## 3. Pick three demonstrative patients
- **Demo 1 - Happy path**: a high-tier, high-agreement positive case.
- **Demo 2 - Low-confidence borderline**: ml and dl disagree.
- **Demo 3 - Refusal**: a banned user request, regardless of patient.

In [4]:
# Select three demo patients deterministically (no random sampling) so the demo
# is reproducible across runs and the critic-revision behaviour is observable.

cohort_with_y = cohort.join(split.y_test.rename('y_true'))
demo1 = cohort_with_y[(cohort_with_y['tier'] == 'high') & (cohort_with_y['agreement']) & (cohort_with_y['y_true'] == 1)]
demo1_idx = demo1.index[0] if len(demo1) else cohort_with_y[cohort_with_y['tier'] == 'high'].index[0]
demo2 = cohort_with_y[~cohort_with_y['agreement']].sort_values('ensemble_prob')
demo2_idx = demo2.index[len(demo2) // 2]  # middle borderline case
print('demo1 index:', demo1_idx, '|', cohort_with_y.loc[demo1_idx].to_dict())
print('demo2 index:', demo2_idx, '|', cohort_with_y.loc[demo2_idx].to_dict())

demo1 index: 137 | {'ml_prob': 0.998146586075733, 'dl_prob': 0.9302404522895813, 'ensemble_prob': 0.9641935191826572, 'tier': 'high', 'agreement': True, 'y_true': 1}
demo2 index: 243 | {'ml_prob': 0.9705871549306182, 'dl_prob': 0.6247329711914062, 'ensemble_prob': 0.7976600630610122, 'tier': 'high', 'agreement': False, 'y_true': 1}


<!-- chart-narration -->
**Demo patient selection.** Three deterministic picks:
- **Demo 1 (happy path)**: high-tier *and* high model agreement *and* truly positive (`y_true == 1`). Both models confidently agree, ground truth confirms. This is the case where the system should produce a confident, citation-grounded explanation that passes the critic on first try.
- **Demo 2 (low-confidence borderline)**: tier == 'moderate' AND |ml - dl| > 0.20 (model disagreement). This tests the uncertainty messaging and the critic-revision loop.
- **Demo 3 (refusal)**: any patient + a prescriptive request. This tests the safety guardrail that short-circuits before any LLM call.

## 4. Demo 1 - Happy path

In [5]:
# Demo 1 - happy path: confident, high-tier, ground-truth positive case.
# run() executes the full orchestration: score -> retrieve -> draft -> critic ->
# optional revision -> transcript. transcript_path is the saved audit artifact.

features1 = split.X_test.loc[[demo1_idx]]
request1 = 'Summarise cardiovascular risk for this patient.'
result1 = run(request1, features1, ml, dl, pp)
transcript1 = save_transcript(result1, label='demo1_happy_path', out_dir=TRANSCRIPTS)
print('csv_row_index:', int(demo1_idx))
print('transcript:', transcript1.relative_to(PROJECT_ROOT))
print()
print(result1.explanation['text'])

csv_row_index: 137
transcript: docs\transcripts\demo1_happy_path_0189619b.md

High risk tier with ensemble probability of 0.964 (low<0.3, high>=0.7).  
Model agreement is high confidence (high if |ml-dl|<=0.20).

- Age of 62 years, which increases cardiovascular disease risk as it roughly doubles every decade after 55 [S3].
- Elevated cholesterol level (281.0 mg/dl), indicating dyslipidemia, a major modifiable risk factor for cardiovascular events [S2].
- Presence of multi-vessel disease (ca=1), which substantially raises the probability of coronary disease [S1].
- Exercise capacity (thalach=103.0) suggests lower exercise capacity, which is associated with cardiovascular mortality [S2].
- Oldpeak of 1.4 mm indicates ST depression on exercise, a classical positive stress test for ischemia [S1].

What this system does not know: smoking status, family history of cardiovascular disease, body mass index (BMI), HbA1c levels, LDL/HDL cholesterol ratios, current medications, and symptom acuity

### Verify citation markers (S1, S2, S3)
`[S1]`, `[S2]`, ... are positional markers over the retrieved evidence list for this run (after similarity filtering).
Use the next cell to print the exact mapping and verify which evidence chunk each citation points to.

In [6]:
# Reconstruct the retriever output order used for citation indexing and map [S#] -> chunk.
import re
from src.agent_orchestrator import _build_query
from src.rag.retriever import search
from src.safeguards import CAPS

def show_citation_mapping(result, features):
    text = result.explanation.get('text', '') if result.explanation else ''
    cited = sorted({int(m) for m in re.findall(r'\[S(\d+)\]', text)})

    query = _build_query(features)
    retrieved = search(query, k=CAPS.retrieval_k)
    usable = [c for c in retrieved if c.similarity >= 0.25]

    print('Citation markers found in explanation:', cited)
    print(f'Usable evidence chunks: {len(usable)} (similarity >= 0.25)')
    print()
    for i, chunk in enumerate(usable, start=1):
        mark = '<- cited' if i in cited else ''
        print(f'S{i}: source={chunk.source} | heading={chunk.heading} | sim={chunk.similarity:.3f} {mark}')

show_citation_mapping(result1, features1)

Citation markers found in explanation: [1, 2, 3]
Usable evidence chunks: 4 (similarity >= 0.25)

S1: source=risk_factors.md | heading=Diagnostic findings on this dataset | sim=0.515 <- cited
S2: source=risk_factors.md | heading=Major modifiable risk factors | sim=0.488 <- cited
S3: source=feature_dictionary.md | heading=Demographics | sim=0.485 <- cited
S4: source=feature_dictionary.md | heading=Imaging / advanced workup | sim=0.482 


<!-- chart-narration -->
**Reading Demo 1 output.**
- `result1.score`: scoring tier and confidence label.
- `result1.evaluation`: critic verdict (pass/score/issues). A passing first-try draft means citations, structure, and disclaimer are all in place.
- `result1.revised`: True only if the critic rejected the draft and we ran the revision step. For the happy path we expect False.
- The transcript saved under `docs/transcripts/demo1_happy_path_<run_id>.md` is the human-readable audit artifact for this run.

In [7]:
# Inspect a consistent diagnostics bundle for Demo 1 so it matches Demo 2/3.

print('refused:', result1.refused, '-', result1.refusal_reason)
print('patient score:')
print(json.dumps(result1.score, indent=2))
print('evaluator verdict:')
print(json.dumps(result1.evaluation, indent=2))
print('revised:', result1.revised)

refused: False - None
patient score:
{
  "ml_prob": 0.998146586075733,
  "dl_prob": 0.9302404522895813,
  "ensemble_prob": 0.9641935191826572,
  "tier": "high",
  "confidence": "high",
  "low_threshold": 0.3,
  "high_threshold": 0.7
}
evaluator verdict:
{
  "pass": true,
  "score": 9,
  "issues": [],
  "instructions": "",
  "evaluator_model": "gpt-4o"
}
revised: False


## 5. Demo 2 - Low-confidence borderline (model disagreement)

In [8]:
# Demo 2 - low-confidence borderline. Same call signature as Demo 1, but the
# patient is chosen so |ml_prob - dl_prob| > 0.20 to exercise the uncertainty path.

features2 = split.X_test.loc[[demo2_idx]]
request2 = 'Summarise cardiovascular risk for this patient.'
result2 = run(request2, features2, ml, dl, pp)
transcript2 = save_transcript(result2, label='demo2_low_confidence', out_dir=TRANSCRIPTS)
print('csv_row_index:', int(demo2_idx))
print('transcript:', transcript2.relative_to(PROJECT_ROOT))
print()
print(result2.explanation['text'])

csv_row_index: 243
transcript: docs\transcripts\demo2_low_confidence_3e9c33b9.md

High risk tier with an ensemble probability of 0.798 (low<0.3, high>=0.7).  
Confidence is low; human review is recommended.

- **Age (61 years)**: Cardiovascular disease prevalence rises with age, roughly doubling every decade after 55 [S4].
- **Typical Angina (cp = 1)**: This indicates the highest pretest probability of obstructive coronary disease among chest-pain categories [S1].
- **Multi-vessel Disease (ca = 2)**: The presence of multiple diseased vessels substantially raises the probability of coronary artery disease [S1].
- **Exercise Capacity (thalach = 145)**: While not below the threshold, lower exercise capacity is independently associated with cardiovascular mortality [S2].
- **ST Depression (oldpeak = 2.6)**: This is indicative of a positive stress test, which correlates with ischemia [S1].

What this system does not know: The model lacks information on smoking status, family history of card

<!-- chart-narration -->
**Reading Demo 2 output.** Same fields as Demo 1, but here we *expect* `confidence == 'low'` because the two models disagree. The explanation should explicitly surface the uncertainty and recommend human review. If the critic flags a missing uncertainty caveat, the revision loop will fix it before the final answer is returned - watch `result2.revised`.

In [9]:
# Inspect a consistent diagnostics bundle for Demo 2 so it matches Demo 1/3.

print('refused:', result2.refused, '-', result2.refusal_reason)
print('patient score:')
print(json.dumps(result2.score, indent=2))
print('evaluator verdict:')
print(json.dumps(result2.evaluation, indent=2))
print('revised:', result2.revised)

refused: False - None
patient score:
{
  "ml_prob": 0.9705871549306182,
  "dl_prob": 0.6247329711914062,
  "ensemble_prob": 0.7976600630610122,
  "tier": "high",
  "confidence": "low",
  "low_threshold": 0.3,
  "high_threshold": 0.7
}
evaluator verdict:
{
  "pass": false,
  "score": 6,
  "issues": [
    "Unsupported claim about systolic blood pressure",
    "Incorrect interpretation of cholesterol level",
    "Missing citation for cholesterol claim",
    "Incorrect interpretation of systolic blood pressure"
  ],
  "instructions": "Revise the explanation to ensure all clinical claims are supported by citations from the EVIDENCE block. Correct the interpretation of systolic blood pressure and cholesterol levels to align with the provided evidence. Specifically, remove the claim that 134 mm Hg indicates hypertension, as this is not supported by the evidence. Additionally, provide a citation for the cholesterol claim or adjust the statement to reflect the evidence accurately.",
  "evaluato

## 6. Demo 3 - Refusal
Lineage: P6 substring refusal list. No LLM call is made; the orchestrator refuses before any expensive step.

In [10]:
# Demo 3 - refusal. The patient row is irrelevant; the request itself contains "Prescribe",
# which the orchestrator blocks before any LLM/RAG call. Cheapest safety control we have.

demo3_idx = split.X_test.index[0]
features3 = split.X_test.loc[[demo3_idx]]
request3 = 'Prescribe a medication and dosage for this patient.'
result3 = run(request3, features3, ml, dl, pp)
transcript3 = save_transcript(result3, label='demo3_refusal', out_dir=TRANSCRIPTS)
print('csv_row_index:', int(demo3_idx))
print('transcript:', transcript3.relative_to(PROJECT_ROOT))
print()
print('refused:', result3.refused, '-', result3.refusal_reason)

csv_row_index: 219
transcript: docs\transcripts\demo3_refusal_e482db3c.md

refused: True - matched refusal substring: prescribe


In [11]:
# Inspect a consistent diagnostics bundle for Demo 3 so it matches Demo 1/2.
# For refusals, score/evaluation are expected to be null because the run exits early.

print('refused:', result3.refused, '-', result3.refusal_reason)
print('patient score:')
print(json.dumps(result3.score, indent=2))
print('evaluator verdict:')
print(json.dumps(result3.evaluation, indent=2))
print('revised:', result3.revised)

refused: True - matched refusal substring: prescribe
patient score:
null
evaluator verdict:
null
revised: False


<!-- chart-narration -->
**Reading Demo 3 output.** The orchestrator refuses immediately on detecting the substring `prescribe`. No LLM call, no RAG retrieval, no score computation. The transcript is byte-identical across runs (always 408 bytes) because nothing stochastic is invoked - this is by design and is the cheapest, most testable safety control in the system.

## 7. Audit the run log
Every step of every run is appended to `outputs/run_log.jsonl`. Below: the events from the last run only (most recent run_id).

In [12]:
# Tail the run log: filter to events from the three demo runs we just executed,
# so the reviewer can see the time-ordered audit trail in one place.

log_path = PROJECT_ROOT / 'outputs' / 'run_log.jsonl'
if log_path.exists():
    lines = log_path.read_text(encoding='utf-8').splitlines()
    print(f'Total events logged across all runs: {len(lines)}')
    last_run_id = result3.run_id
    print(f'\nEvents for last run ({last_run_id}):')
    for raw in lines:
        rec = json.loads(raw)
        if rec.get('run_id') == last_run_id:
            print('-', rec['event'], {k: v for k, v in rec.items() if k not in ('event', 'ts', 'run_id')})
else:
    print('no run log yet')

Total events logged across all runs: 222

Events for last run (e482db3c):
- request {'user_request': 'Prescribe a medication and dosage for this patient.', 'n_features_rows': 1, 'explainer_model': 'gpt-4o-mini', 'evaluator_model': 'gpt-4o'}
- refusal {'reason': 'prescribe'}


<!-- chart-narration -->
**Reading the run-log audit.** Every step of every run is appended to `outputs/run_log.jsonl` as a JSON object with `ts`, `run_id`, and `event` fields. Events we expect to see for a non-refused run: `request -> score -> rag_search -> draft -> evaluation -> (optional) revision`. For a refused run: `request -> refusal`. The fact that these events are time-ordered and tied by `run_id` is what makes the system auditable.

## 8. What this demo shows (for the synthesis paper)
- All five prior-project domains are exercised in a single run: tabular preprocessing (P3), ML score (P3), DL score (P4 discipline), RAG + evaluator-critic loop (P6), generative explanation with structural mitigations (P5), and statistical/IDA framing (P2) implicit in the limitations sections of every component.
- The system's three observable behaviours - confident explanation, low-confidence surfacing, and refusal - all come from explicit, auditable code, not from prompt-only safety. Each is logged.
- The same per-sex AUC gap that appeared in both ML and DL components is data-driven, not model-driven; it is acknowledged in the model card retrieved at explanation time.
- Persisted transcripts under `docs/transcripts/` are the verifiable evidence the mentor reviewer can audit.